# Raw CSV Feature Audit: Events and Usage

This notebook performs a complete feature-level analysis of:
- `data/raw/sample_instance_events.csv`
- `data/raw/sample_instance_usage.csv`

It covers:
1. Dataset shape and preview
2. Column-by-column schema and missingness
3. Full distinct values for each feature (saved to files)
4. Frequency tables for low-cardinality features
5. Numeric distribution summaries

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 180)

RAW_DIR = Path('../data/raw')
OUT_DIR = Path('../data/processed/feature_audit')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    'events': RAW_DIR / 'sample_instance_events.csv',
    'usage': RAW_DIR / 'sample_instance_usage.csv',
}

dfs = {name: pd.read_csv(path) for name, path in DATASETS.items()}

for name, df in dfs.items():
    print(f"{name}: shape={df.shape}, file={DATASETS[name]}")
    display(df.head(5))

events: shape=(10000, 6), file=../data/raw/sample_instance_events.csv


,collection_id,priority,scheduling_class,resource_request_cpus,resource_request_ram,machine_id
0,398622899714,0,0,NaN,NaN,NaN
1,381661839371,0,0,NaN,NaN,NaN
2,399947120663,0,0,NaN,NaN,NaN
3,384546031643,0,0,NaN,NaN,NaN
4,399424363605,0,0,NaN,NaN,NaN


usage: shape=(50000, 5), file=../data/raw/sample_instance_usage.csv


,collection_id,start_time,end_time,average_usage_cpus,average_usage_memory
0,330587191296,2429699000000,2429700000000,0.000000,0.000000
1,330587191296,2418296000000,2418300000000,0.000000,0.000000
2,372912766464,10500000000,10800000000,0.003914,0.003044
3,124264792320,2061000000000,2061300000000,0.000185,0.000597
4,124264792320,949500000000,949800000000,0.000431,0.000646


In [2]:
def feature_profile(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(df)
    for col in df.columns:
        s = df[col]
        non_null = int(s.notna().sum())
        nulls = int(s.isna().sum())
        nunique = int(s.nunique(dropna=True))
        rows.append(
            {
                'column': col,
                'dtype': str(s.dtype),
                'rows': n,
                'non_null': non_null,
                'nulls': nulls,
                'null_pct': (nulls / n) * 100 if n else np.nan,
                'unique_non_null_values': nunique,
                'is_numeric': pd.api.types.is_numeric_dtype(s),
            }
        )
    return pd.DataFrame(rows).sort_values(['is_numeric', 'unique_non_null_values', 'column'], ascending=[False, False, True]).reset_index(drop=True)

profiles = {name: feature_profile(df) for name, df in dfs.items()}

for name, profile in profiles.items():
    print(f"\n=== {name.upper()} FEATURE PROFILE ===")
    display(profile)


=== EVENTS FEATURE PROFILE ===


,column,dtype,rows,non_null,nulls,null_pct,unique_non_null_values,is_numeric
0,collection_id,int64,10000,10000,0,0.00,1681,True
1,machine_id,float64,10000,444,9556,95.56,432,True
2,resource_request_cpus,float64,10000,9986,14,0.14,276,True
3,resource_request_ram,float64,10000,9986,14,0.14,66,True
4,priority,int64,10000,10000,0,0.00,13,True
5,scheduling_class,int64,10000,10000,0,0.00,4,True



=== USAGE FEATURE PROFILE ===


,column,dtype,rows,non_null,nulls,null_pct,unique_non_null_values,is_numeric
0,start_time,int64,50000,50000,0,0.0,18430,True
1,end_time,int64,50000,50000,0,0.0,18337,True
2,average_usage_cpus,float64,50000,50000,0,0.0,4150,True
3,average_usage_memory,float64,50000,50000,0,0.0,3631,True
4,collection_id,int64,50000,50000,0,0.0,1487,True


In [4]:
def distinct_values_table(series: pd.Series) -> pd.DataFrame:
    vc = series.value_counts(dropna=False)
    out = vc.reset_index()
    out.columns = ['value', 'count']
    out['pct'] = (out['count'] / len(series)) * 100
    # Normalize value to string for robust CSV export and easier visual inspection.
    out['value_repr'] = out['value'].map(lambda x: '<NA>' if pd.isna(x) else str(x))
    return out[['value', 'value_repr', 'count', 'pct']]

all_distinct_tables = {}

for ds_name, df in dfs.items():
    ds_dir = OUT_DIR / ds_name
    ds_dir.mkdir(parents=True, exist_ok=True)
    all_distinct_tables[ds_name] = {}

    for col in df.columns:
        table = distinct_values_table(df[col])
        all_distinct_tables[ds_name][col] = table

        out_path = ds_dir / f"{col}__distinct_values.csv"
        table.to_csv(out_path, index=False)

    print(f"Saved full distinct-value tables for {ds_name} in: {ds_dir}")

Saved full distinct-value tables for events in: ../data/processed/feature_audit/events
Saved full distinct-value tables for usage in: ../data/processed/feature_audit/usage


In [5]:
# Display complete value frequencies for low-cardinality columns.
LOW_CARDINALITY_THRESHOLD = 60

for ds_name, df in dfs.items():
    print(f"\n=== {ds_name.upper()} LOW-CARDINALITY FEATURES (<= {LOW_CARDINALITY_THRESHOLD} unique values) ===")
    profile = profiles[ds_name]
    low_card_cols = profile.loc[profile['unique_non_null_values'] <= LOW_CARDINALITY_THRESHOLD, 'column'].tolist()

    if not low_card_cols:
        print('No low-cardinality columns found.')
        continue

    for col in low_card_cols:
        print(f"\n--- {col} ---")
        display(all_distinct_tables[ds_name][col])


=== EVENTS LOW-CARDINALITY FEATURES (<= 60 unique values) ===

--- priority ---


,value,value_repr,count,pct
0,0,0,9094,90.94
1,200,200,511,5.11
2,103,103,181,1.81
3,105,105,60,0.60
4,107,107,51,0.51
5,25,25,40,0.40
6,119,119,25,0.25
7,210,210,15,0.15
8,101,101,9,0.09
9,115,115,7,0.07



--- scheduling_class ---


,value,value_repr,count,pct
0,0,0,7832,78.32
1,2,2,1734,17.34
2,1,1,398,3.98
3,3,3,36,0.36



=== USAGE LOW-CARDINALITY FEATURES (<= 60 unique values) ===
No low-cardinality columns found.


In [6]:
# Numeric summary for all numeric features (distribution-friendly stats).
for ds_name, df in dfs.items():
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"\n=== {ds_name.upper()} NUMERIC SUMMARY ===")
    if not num_cols:
        print('No numeric columns found.')
        continue

    summary = df[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
    summary['nulls'] = df[num_cols].isna().sum()
    summary['null_pct'] = (summary['nulls'] / len(df)) * 100
    display(summary)

# Optional: inspect one high-cardinality column at a time with full frequency table.
# Example:
# display(all_distinct_tables['usage']['start_time'])


=== EVENTS NUMERIC SUMMARY ===


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,nulls,null_pct
collection_id,10000.0,3.302564e+11,1.223754e+11,4.142189e+09,3.951699e+10,3.951700e+10,3.744715e+11,3.813077e+11,3.840762e+11,3.979865e+11,3.997551e+11,4.004619e+11,0,0.00
priority,10000.0,1.422570e+01,4.785888e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+02,2.000000e+02,2.100000e+02,0,0.00
scheduling_class,10000.0,3.974000e-01,7.797012e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00,2.000000e+00,3.000000e+00,0,0.00
resource_request_cpus,9986.0,8.010452e-03,5.984308e-03,0.000000e+00,0.000000e+00,1.863003e-03,3.238678e-03,4.051208e-03,1.417542e-02,1.620483e-02,2.026367e-02,8.630371e-02,14,0.14
resource_request_ram,9986.0,2.990749e-03,1.413807e-03,0.000000e+00,0.000000e+00,2.059937e-04,1.953125e-03,3.906250e-03,3.906250e-03,3.906250e-03,3.906250e-03,3.906250e-03,14,0.14
machine_id,444.0,6.078501e+10,9.580927e+10,2.073360e+07,2.073690e+07,2.127117e+07,1.578714e+09,2.362447e+10,9.203712e+10,3.575159e+11,3.764942e+11,3.839358e+11,9556,95.56



=== USAGE NUMERIC SUMMARY ===


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,nulls,null_pct
collection_id,50000.0,2.263241e+11,1.268798e+11,277.0,4.422663e+09,6.548250e+09,1.056218e+11,2.205859e+11,3.305873e+11,3.817401e+11,3.945132e+11,4.002604e+11,0,0.0
start_time,50000.0,1.338786e+12,7.862192e+11,300000000.0,2.820000e+10,1.245000e+11,6.534000e+11,1.341600e+12,2.015100e+12,2.565600e+12,2.661977e+12,2.678873e+12,0,0.0
end_time,50000.0,1.339028e+12,7.862190e+11,600000000.0,2.850000e+10,1.247872e+11,6.537000e+11,1.341900e+12,2.015158e+12,2.565900e+12,2.662203e+12,2.679000e+12,0,0.0
average_usage_cpus,50000.0,6.496786e-03,1.370868e-02,0.0,0.000000e+00,0.000000e+00,1.993179e-04,4.768372e-04,8.270264e-03,3.030396e-02,4.534973e-02,3.378906e-01,0,0.0
average_usage_memory,50000.0,3.800764e-03,6.597427e-03,0.0,0.000000e+00,9.536743e-07,2.059937e-04,5.474091e-04,4.793167e-03,1.870728e-02,2.481140e-02,8.398438e-02,0,0.0
